---

## **Section 1: Setup and Understanding the Data**

### 🎯 Objective
Import necessary libraries and load the IMDB movie review dataset.

### 📝 Tasks
1. Import all required libraries
2. Check device availability (GPU or CPU)
3. Load the IMDB dataset and convert labels to numeric

### 🤔 Think About It
What makes a review positive vs negative? The words used, right? That's why embeddings are perfect — they capture word meaning!

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f"PyTorch version : {torch.__version__}")
print(f"NumPy version   : {np.__version__}")
print(f"Pandas version  : {pd.__version__}")

PyTorch version : 2.9.1+cpu
NumPy version   : 2.3.4
Pandas version  : 2.3.3


In [2]:
# Check device availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available, using CPU")

Using device: cpu
GPU not available, using CPU


In [3]:
# Load the IMDB movie review dataset
df = pd.read_csv('IMDB Dataset.csv')

# Convert sentiment labels: 'positive' -> 1, 'negative' -> 0
reviews = df['review'].tolist()
labels  = df['sentiment'].map({'positive': 1, 'negative': 0}).tolist()

print("Dataset loaded successfully!")
print(df.head())

Dataset loaded successfully!
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [4]:
# Dataset statistics
total = len(reviews)
n_pos = labels.count(1)
n_neg = labels.count(0)

print(f"Total reviews   : {total:,}")
print(f"Positive reviews: {n_pos:,} ({n_pos/total*100:.1f}%)")
print(f"Negative reviews: {n_neg:,} ({n_neg/total*100:.1f}%)")

print("\n--- Example Positive Review ---")
pos_idx = labels.index(1)
print(reviews[pos_idx][:300], '...')

print("\n--- Example Negative Review ---")
neg_idx = labels.index(0)
print(reviews[neg_idx][:300], '...')

Total reviews   : 50,000
Positive reviews: 25,000 (50.0%)
Negative reviews: 25,000 (50.0%)

--- Example Positive Review ---
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Tru ...

--- Example Negative Review ---
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to ma ...


---

## **Section 2: Text Preprocessing**

### 🎯 Objective
Clean and prepare the text data for our model.

**Steps:**
1. Convert to lowercase
2. Remove punctuation (`re.sub`)
3. Collapse whitespace
4. Split into word tokens
5. **Lemmatize** each token to its base form (see explanation below)
6. Load GloVe embeddings (needed before building vocabulary)
7. Build `word_to_idx` / `idx_to_word` — **only keeping words that exist in GloVe**

### 🤔 Think About It
Why preprocess? Because `"Movie"`, `"movie"`, and `"movie!"` should all be the same word.  
Why lemmatize? Because `"loved"`, `"loves"`, and `"loving"` should all map to `"love"` — a word GloVe actually knows!

### 🔤 What is Lemmatization?

**Lemmatization** reduces a word to its base dictionary form — called the **lemma**.

| Original word | Part of speech | Lemma |
|---|---|---|
| `running`, `ran`, `runs` | verb | `run` |
| `loved`, `loves`, `loving` | verb | `love` |
| `cats`, `cat's` | noun | `cat` |
| `terrible` | adjective | `terrible` *(unchanged)* |

Unlike **stemming** (which just chops off suffixes and can produce non-words like `"happi"`), lemmatization uses a proper dictionary lookup so the result is always a real word.

**Why does this help here?**

GloVe stores vectors for individual word forms. If GloVe has `"love"` but not `"loved"`, that token would be dropped without lemmatization:

```
Without lemmatization:  "loved"  →  not in GloVe  →  dropped (maps to padding index 0)
With    lemmatization:  "loved"  →  "love"         →  found in GloVe ✓
```

By lemmatizing first, far more of our vocabulary matches GloVe — giving the model richer signal for every review.

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

# Download required NLTK data (only needed once; quiet=True suppresses repeated output)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()
print("NLTK WordNet data ready — lemmatizer initialised.")

In [ ]:
def preprocess_text(text):
    """
    Clean, tokenize, and lemmatize text.
    Args:    text (str) – raw review string
    Returns: list of cleaned, lemmatized tokens
    """
    text = text.lower()                       # lowercase
    text = re.sub(r'[^a-z\s]', '', text)     # remove punctuation / digits
    text = re.sub(r'\s+', ' ', text).strip() # collapse extra whitespace
    tokens = text.split()                     # split into word tokens

    # Lemmatize: try verb form first (run, love, watch…), then noun form (cat, film…)
    # This maps "running"→"run", "loved"→"love", "cats"→"cat"
    tokens = [
        lemmatizer.lemmatize(lemmatizer.lemmatize(t, pos='v'), pos='n')
        for t in tokens
    ]
    return tokens

In [6]:
# Apply preprocessing to every review
tokenized_reviews = [preprocess_text(review) for review in reviews]

print(f"Preprocessing complete — {len(tokenized_reviews):,} reviews processed.")
print("\nExample (first 20 tokens of review 0):")
print(tokenized_reviews[0][:20])

Preprocessing complete — 50,000 reviews processed.

Example (first 20 tokens of review 0):
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'youll', 'be', 'hooked', 'they', 'are', 'right', 'as']


### 📥 Load GloVe Embeddings (before building vocabulary)

We load GloVe **now** — before building `word_to_idx` — so that we can filter the vocabulary in a single pass: any token not found in GloVe is simply excluded from the vocabulary and maps to padding index 0 at inference time.

In [ ]:
# Load GloVe embeddings from file
glove_embeddings = {}
glove_file = 'wiki_giga_2024_50_MFT20_vectors_seed_123_alpha_0.75_eta_0.075_combined.txt'

with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        try:
            vector = np.array(values[1:], dtype='float32')
        except ValueError:
            continue  # skip malformed lines
        glove_embeddings[word] = vector

embedding_dim = 50
print(f"GloVe vectors loaded: {len(glove_embeddings):,}")
print(f"Embedding dimension : {embedding_dim}")

In [ ]:
print("\nSample GloVe embedding:")
sample_word = list(glove_embeddings.keys())[0]
print(f"Word: {sample_word}")
print(f"Vector: {glove_embeddings[sample_word]}")

In [ ]:
# Build vocabulary — only keep tokens that have a GloVe vector
# Tokens not in GloVe are silently dropped; they will map to padding index 0
all_tokens   = [token for review in tokenized_reviews for token in review]
unique_words = sorted(set(all_tokens))

glove_vocab  = [word for word in unique_words if word in glove_embeddings]
dropped      = len(unique_words) - len(glove_vocab)

# Index 0 is reserved for padding
word_to_idx = {word: idx + 1 for idx, word in enumerate(glove_vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
vocab_size  = len(word_to_idx) + 1   # +1 for padding index 0

print(f"Unique tokens in corpus  : {len(unique_words):,}")
print(f"Dropped (not in GloVe)   : {dropped:,}  ({dropped/len(unique_words)*100:.1f}%)")
print(f"Vocabulary size          : {vocab_size - 1:,} words  (+1 padding index 0)")

Vocabulary size : 175,892 unique words (including padding index 0)


In [ ]:
# Sample vocabulary mappings
print(f"Total tokens in corpus : {len(all_tokens):,}")
print(f"Unique words           : {len(unique_words):,}")
print(f"After GloVe filter     : {vocab_size - 1:,}")

print("\nSample word → index mappings:")
for word, idx in list(word_to_idx.items())[:10]:
    print(f"  '{word}' → {idx}")

Total tokens in corpus : 11,457,520
Unique words           : 175,891

Sample word → index mappings:
  'a' → 1
  'aa' → 2
  'aaa' → 3
  'aaaaaaaaaaaahhhhhhhhhhhhhh' → 4
  'aaaaaaaargh' → 5
  'aaaaaaah' → 6
  'aaaaaaahhhhhhggg' → 7
  'aaaaagh' → 8
  'aaaaah' → 9
  'aaaaargh' → 10


---

## **Section 3: Loading Pre-trained GloVe Embeddings**

### 🔗 Connecting the Dots — From Word2Vec to GloVe to Sentiment Analysis

In the Word2Vec session you built a `SkipGramModel` from scratch. Here is a quick reminder of exactly what happened inside it:

```
Text corpus  →  (target, context) pairs  →  SkipGramModel
                                               ├── nn.Embedding   ← lookup table of word vectors
                                               └── nn.Linear      ← predicts context word

After training:  model.get_embeddings()  →  a matrix of shape (vocab_size, 50)
                                            each row = the learned vector for one word
```

The model never saw "cat means animal" written anywhere. It learned that **cat** and **dog** are similar **purely because they kept appearing next to the same words** — `the`, `sat`, `on`, `played`, etc.

---

**So what is GloVe?**

GloVe (Global Vectors for Word Representation) is built on exactly the same idea, but trained at a scale that would take months on a laptop:

| | Word2Vec (your version) | GloVe (what we load today) |
|---|---|---|
| Training data | ~200 words (tiny corpus) | 6 billion tokens — all of Wikipedia + Gigaword |
| Vocabulary | ~30 unique words | 400,000+ words |
| Training time | seconds | days on many GPUs |
| Result | a small embedding matrix | a 1 GB text file with one `word vector…` per line |

The `.txt` file you load below is essentially `model.get_embeddings()` from a massive pre-trained model — someone already did all the hard training and shared the weights.

---

**How does this connect to Sentiment Analysis?**

```
Step 1  ── GloVe pre-training (already done for us)
           Billions of words → word vectors that know:
           "excellent" ≈ "fantastic",  "terrible" ≈ "awful"

Step 2  ── Load those vectors into nn.Embedding.from_pretrained()
           Our SentimentClassifier starts with this rich knowledge built-in

Step 3  ── Fine-tune (freeze=False)
           The embeddings adjust slightly during training to better
           capture *sentiment* specifically — not just general meaning

Step 4  ── Average Pooling + Classifier Head
           Collapses all word vectors into one review vector → predict Positive / Negative
```

This is the **transfer learning** pattern — take knowledge learned on a huge general task, then adapt it to your specific smaller task. You will see this same idea powering GPT and every modern language model!

### 🎯 Objective
Build the embedding matrix from the GloVe vectors for our vocabulary.

Since the vocabulary was already filtered to GloVe words in Section 2, this step is simple — every word in `word_to_idx` is **guaranteed** to have a GloVe vector:

- Every word in our vocabulary → its GloVe vector (no random initialisation needed)
- Padding index 0 → zero vector
- Unknown words at inference time → map to index 0 (zero vector)

In [ ]:
# Build embedding matrix — every row comes directly from GloVe (no random initialisation)
embedding_matrix = np.zeros((vocab_size, embedding_dim), dtype='float32')
for word, idx in word_to_idx.items():
    embedding_matrix[idx] = glove_embeddings[word]

print(f"Embedding matrix shape : {embedding_matrix.shape}")
print(f"All {vocab_size - 1:,} vocabulary words are backed by GloVe vectors.")
print("Note: unknown words at inference time map to padding index 0 (zero vector).")

Original vocabulary size      : 175,891
Words dropped (not in GloVe)  : 87,052 (49.5%)
Vocabulary after GloVe filter : 88,839 words
Embedding matrix shape        : (88840, 50)
Note: unknown words will map to padding (index 0) during sequence encoding.


---

## **Section 4: Preparing Data for Training**

### 🎯 Objective
Convert reviews to fixed-length index sequences and split into train / test sets.

**Why fixed length?** Neural networks need consistent input shapes.  
**Padding** — add zeros at the end for short reviews.  
**Truncating** — cut off long reviews at `max_length`.

In [ ]:
# Maximum token length for each review
max_length = 200

def pad_sequence(sequence, max_length):
    """Pad (with zeros) or truncate a token-index list to max_length."""
    if len(sequence) < max_length:
        sequence = sequence + [0] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]
    return sequence

print(f"Max sequence length: {max_length} tokens")

In [ ]:
# Convert every tokenized review to a padded index sequence
sequences = []
for tokens in tokenized_reviews:
    indices = [word_to_idx.get(token, 0) for token in tokens]  # 0 for unknown words
    padded  = pad_sequence(indices, max_length)
    sequences.append(padded)

print(f"Sequences created : {len(sequences):,}")
print(f"Example (first 20 indices): {sequences[0][:20]}")

In [ ]:
# Convert to PyTorch tensors
X = torch.LongTensor(sequences)
y = torch.LongTensor(labels)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Split into train (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {X_train.shape[0]:,}")
print(f"Test samples     : {X_test.shape[0]:,}")
print(f"\nX_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")

---

## **Section 5: Building the Sentiment Classifier**

### 🎯 Objective
Create a neural network that uses GloVe embeddings to classify sentiment.

**Architecture:**
```
Input: [batch_size, max_length]         word index sequences
  ↓  Embedding Layer (freeze=False)
[batch_size, max_length, 50]            one vector per word
  ↓  Mean Pooling  →  [batch_size, 50]  average across all words
  ↓  Linear (50 → 128) + ReLU + Dropout(0.3)
  ↓  Linear (128 → 2)
Output logits: [batch_size, 2]          [negative_score, positive_score]
```

**Why average pooling?** We need ONE vector for the entire review.  
**Why `freeze=False`?** We start with GloVe knowledge and fine-tune for sentiment.

### 🔗 Connecting to Word2Vec — Using Embeddings for a Task

Recall from the Word2Vec session, after training the `SkipGramModel` you extracted the weights with:

```python
learned_vectors = model.get_embeddings()  # shape: (vocab_size, 50)
```

Those weights — one 50-number row per word — **are** the embedding matrix. Here in Section 5 we do the exact mirror image: instead of extracting weights *after* training, we *inject* pre-trained GloVe weights into a brand new `nn.Embedding` layer before training even starts:

```python
self.embedding = nn.Embedding.from_pretrained(
    torch.FloatTensor(embedding_matrix),  # ← the GloVe matrix from Section 3
    freeze=False,   # fine-tune — don't lock the weights
    padding_idx=0   # ignore the all-zero padding rows
)
```

With `freeze=False` the embeddings are **initialised** with rich GloVe knowledge (trained on billions of words) and then **fine-tuned** during our training loop — the network gently nudges word vectors to be better at capturing *sentiment* rather than just general semantic similarity.

This is the **transfer learning** pattern in miniature:
1. **Pre-train** on a huge general task (Word2Vec / GloVe on Wikipedia)
2. **Fine-tune** on a small specific task (IMDB sentiment, 50 k reviews)

You will see this same idea at the heart of BERT, GPT, and every modern language model.

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super(SentimentClassifier, self).__init__()

        vocab_sz, embed_dim = embedding_matrix.shape

        # Pre-trained GloVe embedding layer (fine-tunable)
        self.embedding = nn.Embedding.from_pretrained(
            torch.FloatTensor(embedding_matrix),
            freeze=False,       # allow fine-tuning during training
            padding_idx=0       # ignore padding tokens in gradients
        )

        # Classifier head
        self.fc1     = nn.Linear(embed_dim, 128)
        self.fc2     = nn.Linear(128, 2)
        self.dropout = nn.Dropout(0.3)
        self.relu    = nn.ReLU()

    def forward(self, x):
        # x: (batch_size, max_length)
        embedded = self.embedding(x)            # (batch, max_length, embed_dim)
        pooled   = torch.mean(embedded, dim=1)  # (batch, embed_dim)  — mean pooling
        x = self.relu(self.fc1(pooled))         # (batch, 128)
        x = self.dropout(x)
        out = self.fc2(x)                       # (batch, 2)
        return out

In [ ]:
# Instantiate the model and move to device
model = SentimentClassifier(embedding_matrix).to(device)

print("Model Architecture:")
print("=" * 60)
print(model)
print("=" * 60)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

---

## **Section 6: Training Setup**

In [ ]:
# Loss function — CrossEntropyLoss is perfect for 2-class classification
criterion  = nn.CrossEntropyLoss()

# Optimizer — Adam with a sensible learning rate
optimizer  = torch.optim.Adam(model.parameters(), lr=0.001)

# How many passes over the training data
num_epochs = 5

In [ ]:
# Print training configuration
print("Training Configuration")
print("=" * 40)
print(f"Loss function : CrossEntropyLoss")
print(f"Optimizer     : Adam  (lr=0.001)")
print(f"Epochs        : {num_epochs}")
print(f"Device        : {device}")
print(f"Train samples : {X_train.shape[0]:,}")
print(f"Test samples  : {X_test.shape[0]:,}")
print(f"Batch size    : 256")

---

## **Section 7: The Training Loop**

### 🎯 Objective
Train the classifier and track loss + accuracy each epoch.

**Training loop pattern** (same structure you saw in previous sessions):
1. `model.train()` → enable dropout
2. Mini-batch forward pass → compute loss → backprop → update weights
3. `model.eval()` + `torch.no_grad()` → evaluate on test set

**Expected results:** 85–90%+ test accuracy on the full IMDB dataset after a few epochs.

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test,
                criterion, optimizer, num_epochs, batch_size=256):
    """
    Train the sentiment classifier and return metric history.
    Returns a dict with keys: train_loss, train_acc, test_loss, test_acc.
    """
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss' : [], 'test_acc' : []
    }

    n_train = X_train.shape[0]

    for epoch in range(1, num_epochs + 1):

        # ── Training phase ─────────────────────────────────────────
        model.train()
        train_loss    = 0.0
        train_correct = 0

        for i in range(0, n_train, batch_size):
            X_batch = X_train[i : i + batch_size].to(device)
            y_batch = y_train[i : i + batch_size].to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss    = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item() * X_batch.size(0)
            _, predicted   = torch.max(outputs, 1)
            train_correct += (predicted == y_batch).sum().item()

        avg_train_loss = train_loss / n_train
        avg_train_acc  = train_correct / n_train

        # ── Validation phase ────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            X_t    = X_test.to(device)
            y_t    = y_test.to(device)
            t_out  = model(X_t)
            t_loss = criterion(t_out, y_t).item()
            _, t_pred = torch.max(t_out, 1)
            t_acc  = (t_pred == y_t).sum().item() / y_t.size(0)

        history['train_loss'].append(avg_train_loss)
        history['train_acc' ].append(avg_train_acc)
        history['test_loss' ].append(t_loss)
        history['test_acc'  ].append(t_acc)

        print(f"Epoch [{epoch}/{num_epochs}]  "
              f"Train Loss: {avg_train_loss:.4f}  Train Acc: {avg_train_acc:.4f}  |  "
              f"Test Loss: {t_loss:.4f}  Test Acc: {t_acc:.4f}")

    return history

In [ ]:
# Train the model
print("Starting training...")
print("=" * 80)
history = train_model(
    model, X_train, y_train, X_test, y_test,
    criterion, optimizer, num_epochs=num_epochs
)
print("=" * 80)
print("Training complete!")

---

## **Section 8: Visualising Training Progress**

In [ ]:
# Plot loss and accuracy curves side by side
epochs_range = range(1, num_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(epochs_range, history['train_loss'], marker='o', label='Train Loss')
axes[0].plot(epochs_range, history['test_loss'],  marker='s', label='Test Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(epochs_range, history['train_acc'], marker='o', label='Train Accuracy')
axes[1].plot(epochs_range, history['test_acc'],  marker='s', label='Test Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Sentiment Classifier — Training Progress', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final metrics summary
best_epoch = history['test_acc'].index(max(history['test_acc'])) + 1
print(f"Final Training Accuracy : {history['train_acc'][-1]:.4f}")
print(f"Final Test Accuracy     : {history['test_acc'][-1]:.4f}")
print(f"Best Test Accuracy      : {max(history['test_acc']):.4f}  (epoch {best_epoch})")

---

## **Section 9: Detailed Evaluation**

### 🎯 Objective
Go beyond accuracy — look at precision, recall, F1, and where the model goes wrong.

| Metric | Meaning |
|---|---|
| **Precision** | Of all predicted Positive, how many really were? |
| **Recall** | Of all actual Positives, how many did we catch? |
| **F1-Score** | Harmonic mean of precision and recall |
| **Confusion Matrix** | Where the model confuses the two classes |

In [ ]:
# Get predictions on the full test set
model.eval()
with torch.no_grad():
    outputs      = model(X_test.to(device))
    _, predicted = torch.max(outputs, 1)
    y_pred = predicted.cpu().numpy()
    y_true = y_test.cpu().numpy()

print(f"Test set size   : {len(y_true):,}")
print(f"Overall Accuracy: {accuracy_score(y_true, y_pred):.4f}")

In [ ]:
# Precision / Recall / F1 per class
print(classification_report(y_true, y_pred, target_names=['Negative', 'Positive']))

In [ ]:
# Confusion matrix heatmap
cm           = confusion_matrix(y_true, y_pred)
class_names  = ['Negative', 'Positive']

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)

ax.set_xticks([0, 1]); ax.set_xticklabels(class_names)
ax.set_yticks([0, 1]); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix')

threshold = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]),
                ha='center', va='center', fontsize=14,
                color='white' if cm[i, j] > threshold else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# Analyse misclassified reviews
# Recover the original test-set indices (same random_state=42)
_, test_indices = train_test_split(
    range(len(reviews)), test_size=0.2, random_state=42
)

misclassified = [i for i, (p, t) in enumerate(zip(y_pred, y_true)) if p != t]

print(f"Misclassified: {len(misclassified):,} out of {len(y_true):,}")
print("\n--- Example Misclassified Reviews ---\n")

for idx in misclassified[:3]:
    orig_idx  = test_indices[idx]
    true_lbl  = 'Positive' if y_true[idx] == 1 else 'Negative'
    pred_lbl  = 'Positive' if y_pred[idx] == 1 else 'Negative'
    print(f"Review    : {reviews[orig_idx][:220]}...")
    print(f"True label: {true_lbl}  |  Predicted: {pred_lbl}")
    print()

---

## **Section 10: Testing on New Reviews**

### 🎯 Objective
Create a reusable `predict_sentiment()` function and test on brand-new reviews.

### 🤔 Think About It
- Does your model generalise well to different writing styles?
- What about sarcasm? Mixed reviews? Very short reviews?

In [ ]:
def predict_sentiment(review_text, model, word_to_idx, max_length):
    """
    Predict the sentiment of a new movie review.

    Args:
        review_text : raw review string
        model       : trained SentimentClassifier
        word_to_idx : vocabulary mapping
        max_length  : fixed sequence length used during training

    Returns:
        (sentiment_label, confidence_score)
    """
    model.eval()

    # Preprocess
    tokens  = preprocess_text(review_text)
    indices = [word_to_idx.get(token, 0) for token in tokens]  # 0 for OOV
    padded  = pad_sequence(indices, max_length)

    # Tensor with batch dimension
    x = torch.LongTensor([padded]).to(device)

    with torch.no_grad():
        output     = model(x)
        probs      = torch.softmax(output, dim=1)
        predicted  = torch.argmax(probs, dim=1).item()
        confidence = probs[0][predicted].item()

    sentiment = "Positive 😊" if predicted == 1 else "Negative 😞"
    return sentiment, confidence

In [ ]:
# Test on new reviews (never seen during training)
new_reviews = [
    # Clearly positive
    "This film was absolutely breathtaking. The acting, the music, everything was perfect!",
    "One of the best movies I have seen in years. Highly recommend to everyone!",
    "A masterpiece of storytelling — emotional, funny, and deeply moving.",
    # Clearly negative
    "Terrible plot, awful acting. I walked out after 30 minutes.",
    "A complete waste of time and money. Nothing makes any sense.",
    "The worst film I have ever seen. Boring from start to finish.",
    # Ambiguous / tricky
    "It was okay. Not great, not terrible — just very average.",
    "I did not hate it, but I probably would not watch it again.",
]

print(f"{'Review (truncated)':<58} | {'Prediction':<15} | Confidence")
print("-" * 95)
for rev in new_reviews:
    sentiment, conf = predict_sentiment(rev, model, word_to_idx, max_length)
    short = rev[:55] + '...' if len(rev) > 58 else rev
    print(f"{short:<58} | {sentiment:<15} | {conf:.2%}")

In [ ]:
# Interactive mode — type your own review!
user_review = input("Enter a movie review: ")
if user_review.strip():
    sentiment, conf = predict_sentiment(user_review, model, word_to_idx, max_length)
    print(f"\nSentiment  : {sentiment}")
    print(f"Confidence : {conf:.2%}")

---

## **Section 12: Saving Your Model**

### 🎯 Objective
Persist the trained model so it can be reloaded later without retraining.

A **checkpoint** bundles:
- Model weights (`state_dict`)
- Vocabulary mappings
- Hyperparameters (embedding dim, max_length, etc.)
- Training history

In [ ]:
# Build a checkpoint with everything needed to reload the model
checkpoint = {
    'model_state_dict'    : model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'word_to_idx'         : word_to_idx,
    'idx_to_word'         : idx_to_word,
    'max_length'          : max_length,
    'vocab_size'          : vocab_size,
    'embedding_dim'       : embedding_dim,
    'embedding_matrix'    : embedding_matrix,
    'history'             : history,
}

torch.save(checkpoint, 'sentiment_classifier.pth')
print("Model saved  →  sentiment_classifier.pth")

In [ ]:
def load_sentiment_model(path):
    """
    Load a saved SentimentClassifier checkpoint.

    Returns:
        (model, word_to_idx, max_length)
    """
    ckpt = torch.load(path, map_location=device)

    loaded_model = SentimentClassifier(ckpt['embedding_matrix']).to(device)
    loaded_model.load_state_dict(ckpt['model_state_dict'])
    loaded_model.eval()

    return loaded_model, ckpt['word_to_idx'], ckpt['max_length']


# Verify the checkpoint loads and produces correct predictions
loaded_model, loaded_w2i, loaded_maxlen = load_sentiment_model('sentiment_classifier.pth')
sent, conf = predict_sentiment(
    "This is a fantastic movie — I loved every minute!",
    loaded_model, loaded_w2i, loaded_maxlen
)
print(f"Test prediction from re-loaded model: {sent} ({conf:.2%})")
print("\nAll done! Your sentiment classifier is trained, evaluated, and saved.")